# VCD on POPE — Colab runner (A100)

**Budget rules:** Colab bills compute units for every minute a runtime is connected, *including idle time*.
Run the setup cells, start the main cell, keep the tab open; the main cell **flushes Drive and disconnects the runtime** when it finishes.
All outputs are written directly to Google Drive, so an interrupted run is resumed by running sections 1–3 again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup. After the pip cell: *Runtime → Restart session* once, then run section 1 again from the top.

In [ ]:
REPO_URL = "https://github.com/mirhanayd/yzm5110-vcd-pope.git"
DRIVE = "/content/drive/MyDrive/yzm5110_vcd_results"
REPO = "/content/vcd-pope"

from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull", "--ff-only"], check=True)

# every output folder is a symlink into Drive (done in Python: no shell variable expansion pitfalls)
links = {f"results/{d}": f"{DRIVE}/{d}" for d in ("runs", "timing", "logs", "jobs", "tables")}
links["data/images"] = f"{DRIVE}/images"
for local, target in links.items():
    os.makedirs(target, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.exists(local):
        shutil.rmtree(local)
    os.symlink(target, local)
for local, target in links.items():   # hard check: refuse to continue if anything would be written off-Drive
    assert os.path.realpath(local) == os.path.realpath(target), (local, os.path.realpath(local))
print("all outputs go to Drive:", sorted(links))
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt 2>&1 | grep -v "dependency conflicts\|requires huggingface-hub" || true
import transformers; print(transformers.__version__)   # must print 4.51.3, otherwise restart the session

## 2. Data (images are kept on Drive; later sessions skip the download)

In [ ]:
!python scripts/download_data.py

## 3. All experiments in one session (~5 h on A100, ~45 units)
Jobs run in priority order (core reproduction first). Progress: run `!tail -n 2 results/logs/all_worker*.log` in another cell.
If the session dies, run sections 1–3 again: finished questions are skipped.

In [ ]:
!TAU=0.6 SKIP=0.9581 WORKERS=4 bash scripts/experiments.sh all
!bash scripts/experiments.sh tables
!ls results/runs | wc -l; ls results/timing | wc -l
import time; from google.colab import drive, runtime
drive.flush_and_unmount()   # write everything to Drive BEFORE the VM is released
time.sleep(30)
runtime.unassign()          # stop billing

## 4. Tables only (CPU runtime is enough)

In [ ]:
!bash scripts/experiments.sh tables
!cat results/tables/main.md results/tables/timing_main.md